MovieLens 100k – Memory‑Based CF Experiments
-------------------------------------------------------------
This notebook‑style Python script implements the full experimental
pipeline described in the thesis:
  * Data loading & preprocessing (MovieLens‑100k)
  * Similarity functions: Cosine, Adjusted Cosine, Pearson, Jaccard
  * k‑NN CF (user‑based & item‑based)
  * Baseline model‑based method: Funk‑SVD (Surprise)
  * Evaluation: 5‑fold user‑stratified CV, rating (RMSE/MAE) &
    ranking (Precision@k, Recall@k, nDCG@k, Coverage, Diversity)
  * Significance testing (paired t‑test / Wilcoxon)
-------------------------------------------------------------
All heavy lifting happens in pure NumPy/Pandas + Surprise (for Funk‑SVD).
To run as a Jupyter notebook, open this file with JupyterLab; each "%%"
cell delimiter will be recognised as a cell boundary.
-------------------------------------------------------------

## Imports & Globals

In [179]:
import pathlib
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from typing import List, Dict, Set, Tuple, Literal, Optional, Union
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import ttest_rel, wilcoxon, pearsonr, ConstantInputWarning
import scipy.spatial.distance as dist
import warnings
from joblib import Parallel, delayed
from scipy.sparse import csr_matrix

# Optional – Surprise for Funk‑SVD baseline
try:
    from surprise import Dataset, Reader, SVD
    from surprise.model_selection import train_test_split as sp_train_test_split
except ImportError:
    SVD = None  # graceful degradation

# Constants
DATA_DIR = pathlib.Path("ml-latest-small")
RATING_SCALE = (1, 5)
SEED = 42
np.random.seed(SEED)

## Data Loading

In [180]:
def load_movielens_latest_small(path: Union[str, pathlib.Path] = DATA_DIR) -> pd.DataFrame:
    """Load MovieLens ratings dataset and return a DataFrame with renamed columns."""
    path = pathlib.Path(path)
    ratings_path = path / "ratings.csv"
    if not ratings_path.exists():
        raise FileNotFoundError(
            f"File '{ratings_path}' not found. Download from "
            "https://grouplens.org/datasets/movielens/latest/ and unzip to 'ml-latest-small/'."
        )
    df = pd.read_csv(ratings_path, usecols=["userId", "movieId", "rating", "timestamp"])
    df.rename(columns={"userId": "user", "movieId": "item"}, inplace=True)
    return df

# Load data
ratings_df = load_movielens_latest_small()
print(f"Loaded {len(ratings_df):,} ratings → {ratings_df['user'].nunique()} users × {ratings_df['item'].nunique()} items")

# TEMPORARY: reduce dataset for debugging
#ratings_df = ratings_df.sample(frac=0.1, random_state=SEED).reset_index(drop=True)
#print(f"Sampled {len(ratings_df):,} ratings → {ratings_df['user'].nunique()} users × {ratings_df['item'].nunique()} items")

Loaded 100,836 ratings → 610 users × 9724 items


## Utility: Sparse Ratings Matrix

In [181]:
def df_to_matrix(df: pd.DataFrame) -> Tuple[np.ndarray, Dict[int, int], Dict[int, int]]:
    """Return (rating_matrix, uid_map, iid_map).
    rating_matrix shape = (n_users, n_items) with float32, NaN for missing."""
    uids = pd.Index(sorted(df["user"].unique()))
    iids = pd.Index(sorted(df["item"].unique()))
    uid_map = {uid: i for i, uid in enumerate(uids)}
    iid_map = {iid: i for i, iid in enumerate(iids)}

    mat = np.full((len(uids), len(iids)), np.nan, dtype=np.float32)
    mat[df["user"].map(uid_map), df["item"].map(iid_map)] = df["rating"].astype(np.float32).values

    return mat, uid_map, iid_map

## Similarity Functions
Each returns a float in $[-1, 1]$ (except Jaccard $[0, 1]$)

In [182]:
SimMetric = Literal["cosine", "adj_cosine", "pearson", "jaccard"]

def cosine_sim(x: np.ndarray, y: np.ndarray) -> float:
    mask = ~np.isnan(x) & ~np.isnan(y)
    if mask.sum() == 0:
        return 0.0
    return cosine_similarity(x[mask].reshape(1, -1), y[mask].reshape(1, -1))[0, 0]


def adj_cosine_sim(x: np.ndarray, y: np.ndarray, item_means: np.ndarray) -> float:
    mask = ~np.isnan(x) & ~np.isnan(y)
    if mask.sum() == 0:
        return 0.0
    assert len(item_means) == len(x), f"item_means shape {item_means.shape} does not match x {x.shape}"
    xc = x[mask] - item_means[mask]
    yc = y[mask] - item_means[mask]
    if np.all(xc == 0) or np.all(yc == 0):
        return 0.0
    return cosine_similarity(xc.reshape(1, -1), yc.reshape(1, -1))[0, 0]


def pearson_sim(x: np.ndarray, y: np.ndarray) -> float:
    mask = ~np.isnan(x) & ~np.isnan(y)
    if mask.sum() < 2:
        return 0.0
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=ConstantInputWarning)
            r, _ = pearsonr(x[mask], y[mask])
            return r if not np.isnan(r) else 0.0
    except Exception:
        return 0.0


def jaccard_sim(x: np.ndarray, y: np.ndarray) -> float:
    x_bin = ~np.isnan(x)
    y_bin = ~np.isnan(y)
    return 1 - dist.jaccard(x_bin, y_bin) if np.any(x_bin | y_bin) else 0.0


SIM_FUNCS = {
    "cosine": cosine_sim,
    "adj_cosine": adj_cosine_sim,
    "pearson": pearson_sim,
    "jaccard": jaccard_sim,
}

## Ranking Metrics

In [183]:
def ranking_metrics(recs: List[List[int]], ground_truth: List[Set[int]], k: int = 10) -> Dict[str, float]:
    precisions, recalls, ndcgs = [], [], []
    recommended_items = set()
    for rec, gt in zip(recs, ground_truth):
        if not gt:
            continue
        top_k = rec[:k]
        hits = [item in gt for item in top_k]
        num_hits = sum(hits)
        precisions.append(num_hits / k)
        recalls.append(num_hits / len(gt))
        dcg = sum(int(hit) / math.log2(rank + 2) for rank, hit in enumerate(hits))
        ideal_dcg = sum(1 / math.log2(rank + 2) for rank in range(min(len(gt), k)))
        ndcgs.append(dcg / ideal_dcg if ideal_dcg > 0 else 0)
        recommended_items.update(top_k)
    return {
        "precision@k": np.mean(precisions) if precisions else 0.0,
        "recall@k": np.mean(recalls) if recalls else 0.0,
        "ndcg@k": np.mean(ndcgs) if ndcgs else 0.0,
        "coverage@k": len(recommended_items)
    }

## k‑NN CF Class

In [184]:
class KNNRecommender:
    def __init__(self, kind: Literal["user", "item"], metric: SimMetric = "cosine", k: int = 20):
        self.kind = kind
        self.metric = metric
        self.k = k
        self.sim_matrix: Optional[np.ndarray] = None
        self.rating_matrix: Optional[np.ndarray] = None
        self.item_means: Optional[np.ndarray] = None

    def fit(self, rating_matrix: np.ndarray):
        self.rating_matrix = rating_matrix
        axis = 0 if self.kind == "user" else 1
        n = rating_matrix.shape[axis]

        self.sim_matrix = np.full((n, n), np.nan, dtype=np.float32)

        # Compute item_means such that it's aligned with the input vectors
        if self.metric == "adj_cosine":
            # If comparing rows (users), item_means must match columns (items)
            self.item_means = np.nanmean(rating_matrix, axis=0 if self.kind == "user" else 1)

        for i in range(n):
            vec_i = self._get_vector(i)
            for j in range(i + 1, n):
                vec_j = self._get_vector(j)
                if self.metric == "adj_cosine":
                    sim = SIM_FUNCS[self.metric](vec_i, vec_j, self.item_means)
                else:
                    sim = SIM_FUNCS[self.metric](vec_i, vec_j)
                self.sim_matrix[i, j] = self.sim_matrix[j, i] = sim
        return self

    def _get_vector(self, idx: int) -> np.ndarray:
        return self.rating_matrix[idx, :] if self.kind == "user" else self.rating_matrix[:, idx]

    def _get_neighbors(self, idx: int, rated_mask: np.ndarray) -> List[int]:
        sims = self.sim_matrix[idx].copy()
        sims[idx] = -np.inf
        candidate_idx = np.where(rated_mask)[0]
        if len(candidate_idx) == 0:
            return []
        top_k_idx = np.argsort(sims[candidate_idx])[-self.k:]
        return candidate_idx[top_k_idx[np.argsort(sims[candidate_idx][top_k_idx])[::-1]]]

    def predict(self, user_idx: int, item_idx: int) -> float:
        if self.kind == "user":
            rated_mask = ~np.isnan(self.rating_matrix[:, item_idx])
            neighbors = self._get_neighbors(user_idx, rated_mask)
            sims = self.sim_matrix[user_idx, neighbors]
            ratings = self.rating_matrix[neighbors, item_idx]
        else:
            rated_mask = ~np.isnan(self.rating_matrix[user_idx, :])
            neighbors = self._get_neighbors(item_idx, rated_mask)
            sims = self.sim_matrix[item_idx, neighbors]
            ratings = self.rating_matrix[user_idx, neighbors]
        if len(neighbors) == 0 or np.all(sims == 0):
            return np.nanmean(self.rating_matrix[user_idx, :])
        return np.dot(sims, ratings) / np.sum(np.abs(sims))

## Cross‑Validation Runner

In [185]:
def run_experiments(rating_df: pd.DataFrame, ks: List[int], metrics: List[SimMetric], kind: Literal["user", "item"] = "user", topk: int = 10, n_splits: int = 5) -> pd.DataFrame:
    def evaluate(metric, k):
        print(f"Evaluating for kind={kind}, topk={topk}, metric={metric}, k={k}")
        start_eval = time.time()
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
        rmses, maes, precisions, recalls, ndcgs, coverages = [], [], [], [], [], []

        for train_idx, test_idx in kf.split(rating_df):
            train_df = rating_df.iloc[train_idx]
            test_df = rating_df.iloc[test_idx]
            R_train, uid_map, iid_map = df_to_matrix(train_df)
            model = KNNRecommender(kind=kind, metric=metric, k=k).fit(R_train)

            # Build ground truth from test set
            test_users_raw = test_df["user"].map(uid_map)
            test_items_raw = test_df["item"].map(iid_map)
            test_ratings = test_df["rating"]

            user_to_true = defaultdict(set)
            for u, i in zip(test_users_raw, test_items_raw):
                if not pd.isna(u) and not pd.isna(i):
                    user_to_true[int(u)].add(int(i))

            # Build training item history per user
            train_users = train_df["user"].map(uid_map)
            train_items = train_df["item"].map(iid_map)
            user_rated_items = defaultdict(set)
            for u, i in zip(train_users, train_items):
                if not pd.isna(u) and not pd.isna(i):
                    user_rated_items[int(u)].add(int(i))

            # Generate top-N recommendations
            user_to_pred = defaultdict(list)
            y_true, y_pred = [], []
            all_items = set(range(R_train.shape[1]))

            for u in user_to_true:
                unseen_items = all_items - user_rated_items[u]
                preds = []
                for i in unseen_items:
                    pred = model.predict(u, i)
                    if not np.isnan(pred):
                        preds.append((pred, i))
                top_preds = sorted(preds, reverse=True)[:topk]
                user_to_pred[u] = top_preds

            # Also compute RMSE/MAE for items in test set
            for u, i, r in zip(test_users_raw, test_items_raw, test_ratings):
                if pd.isna(u) or pd.isna(i):
                    continue
                u, i = int(u), int(i)
                pred = model.predict(u, i)
                if not np.isnan(pred):
                    y_true.append(r)
                    y_pred.append(pred)

            # Aggregate metrics
            if y_true:
                rmses.append(math.sqrt(mean_squared_error(y_true, y_pred)))
                maes.append(mean_absolute_error(y_true, y_pred))

            recs = [[i for _, i in user_to_pred[u]] for u in user_to_pred]
            ground_truth = [user_to_true[u] for u in user_to_pred]

            rank = ranking_metrics(recs, ground_truth, k=topk)
            precisions.append(rank["precision@k"])
            recalls.append(rank["recall@k"])
            ndcgs.append(rank["ndcg@k"])
            coverages.append(rank["coverage@k"] / rating_df["item"].nunique())

        print(f"Done metric={metric}, k={k} in {time.time() - start_eval:.2f}s")
        return {
            "kind": kind, "metric": metric, "k": k,
            "rmse": np.nanmean(rmses), "mae": np.nanmean(maes),
            "precision@k": np.nanmean(precisions), "recall@k": np.nanmean(recalls),
            "ndcg@k": np.nanmean(ndcgs), "coverage@k": np.nanmean(coverages)
        }

    import itertools
    results = Parallel(n_jobs=-1)(delayed(evaluate)(metric, k) for metric, k in itertools.product(metrics, ks))
    return pd.DataFrame(results)


## Funk‑SVD Baseline

In [186]:
def run_funk_svd(rating_df: pd.DataFrame, n_epochs: int = 20) -> float:
    if SVD is None:
        print("Surprise not installed – skipping Funk‑SVD baseline.")
        return np.nan

    reader = Reader(rating_scale=RATING_SCALE)
    data = Dataset.load_from_df(rating_df[["user", "item", "rating"]], reader)
    trainset, testset = sp_train_test_split(data, test_size=0.2, random_state=SEED)

    algo = SVD(n_epochs=n_epochs, random_state=SEED)
    algo.fit(trainset)
    predictions = algo.test(testset)

    return math.sqrt(mean_squared_error(
        [p.r_ui for p in predictions],
        [p.est for p in predictions]
    ))


## Significance Testing

In [187]:
def paired_significance(
    scores_a: List[float], 
    scores_b: List[float], 
    test: Literal["ttest", "wilcoxon"] = "ttest"
) -> float:
    """Return p-value from paired significance test."""
    if len(scores_a) != len(scores_b):
        raise ValueError("Score lists must have the same length")
    if test == "ttest":
        stat, p = ttest_rel(scores_a, scores_b, nan_policy="omit")
    else:
        stat, p = wilcoxon(scores_a, scores_b)
    return p


## Visualization

In [188]:
def plot_metric_comparison(df: pd.DataFrame, metric_name: str):
    kinds = df['kind'].unique()
    n_cols = len(kinds)

    fig, axes = plt.subplots(1, n_cols, figsize=(6 * n_cols, 5), sharey=True)

    if n_cols == 1:
        axes = [axes]

    for ax, kind in zip(axes, kinds):
        sub = df[df["kind"] == kind]
        for metric in sorted(sub["metric"].unique()):
            metric_data = sub[sub["metric"] == metric].sort_values("k")
            ax.plot(metric_data["k"], metric_data[metric_name], marker="o", label=metric)
        ax.set_title(f"{kind.capitalize()}-based CF")
        ax.set_xlabel("Neighborhood Size (k)")
        ax.set_ylabel(metric_name)
        ax.legend()
        ax.grid(True)

    plt.suptitle(f"{metric_name} vs Neighborhood Size (k)")
    plt.tight_layout()
    plt.show()

def plot_rmse_comparison(df: pd.DataFrame):
    kinds = df['kind'].unique()
    n_cols = len(kinds)

    fig, axes = plt.subplots(1, n_cols, figsize=(6 * n_cols, 5), sharey=True)

    if n_cols == 1:
        axes = [axes]  # make iterable if only one kind

    for ax, kind in zip(axes, kinds):
        sub = df[df["kind"] == kind]
        for metric in sorted(sub["metric"].unique()):
            metric_data = sub[sub["metric"] == metric].sort_values("k")
            ax.plot(metric_data["k"], metric_data["rmse"], marker="o", label=metric)
        ax.set_title(f"{kind.capitalize()}-based CF")
        ax.set_xlabel("Neighborhood Size (k)")
        ax.set_ylabel("RMSE")
        ax.legend()
        ax.grid(True)

    plt.suptitle("RMSE vs Neighborhood Size (k) for User vs Item-based CF")
    plt.tight_layout()
    plt.show()

## Execution

In [189]:
if __name__ == "__main__":
    import time
    pd.set_option('display.max_columns', None) 

    start = time.time()
    print("Running experiments...")

    grid_user = run_experiments(
        ratings_df,
        ks=[10, 20, 40],
        metrics=["cosine", "adj_cosine", "pearson", "jaccard"],
        kind="user",
        n_splits=3  # reduce CV folds for speed
    )
    print("\nUser-based CF Results:\n", grid_user)

    grid_item = run_experiments(
        ratings_df,
        ks=[10, 20, 40],
        metrics=["cosine", "adj_cosine", "pearson", "jaccard"],
        kind="item",
        n_splits=3
    )
    print("\nItem-based CF Results:\n", grid_item)

    grid_df = pd.concat([grid_user, grid_item], ignore_index=True)
    print("\nCombined Results:\n", grid_df)

    print("Experiments complete in {:.2f}s".format(time.time() - start))

    # Optional baseline
    baseline_rmse = run_funk_svd(ratings_df)
    print(f"Funk‑SVD baseline RMSE: {baseline_rmse:.4f}")

    # Plot results
    plot_rmse_comparison(grid_df)
    for metric in ["mae", "precision@k", "recall@k", "ndcg@k", "coverage@k"]:
        plot_metric_comparison(grid_df, metric)


Running experiments...

User-based CF Results:
     kind      metric   k      rmse       mae  precision@k  recall@k    ndcg@k  \
0   user      cosine  10  0.986252  0.758308     0.000820  0.000170  0.000920   
1   user      cosine  20  0.973317  0.749764     0.000820  0.000170  0.000920   
2   user      cosine  40  0.971006  0.749448     0.000820  0.000170  0.000920   
3   user  adj_cosine  10  2.447774  1.612987     0.000437  0.000031  0.000554   
4   user  adj_cosine  20  2.903394  2.119659     0.000437  0.000031  0.000554   
5   user  adj_cosine  40  3.430638  2.813124     0.000437  0.000031  0.000554   
6   user     pearson  10  1.386128  0.937710     0.000601  0.000168  0.000847   
7   user     pearson  20  1.529817  1.038255     0.000601  0.000168  0.000871   
8   user     pearson  40  1.741175  1.221039     0.000546  0.000059  0.000754   
9   user     jaccard  10  0.980680  0.758975     0.000492  0.000066  0.000632   
10  user     jaccard  20  0.971974  0.751942     0.000492  0.

KeyboardInterrupt: 